# Trino E2E Test

Comprehensive Trino validation for the current BERDL stack.

This notebook uses Spark only to seed and clean test data because Trino is intentionally read-only. The Trino checks cover the legacy Delta/Hive path and the Polaris/Iceberg path with the shared Spark/Trino catalog alias contract.

**Expected catalog forms:**
- Legacy Delta/Hive: `u_<username>.<schema>.<table>` in Trino
- Personal Iceberg: `<username>.<namespace>.<table>` in Spark and Trino
- Tenant Iceberg: `<tenant>.<namespace>.<table>` in Spark and Trino
- Spark-only personal convenience alias: `my.<namespace>.<table>`; this should not exist in Trino


## Test Matrix

| Area | Coverage |
|---|---|
| Environment | Required user, MinIO, Polaris, and Trino settings |
| Catalogs | Dynamic Delta/Hive catalog plus Polaris personal and tenant catalogs |
| Legacy Delta | Trino reads a Spark-seeded Delta table through `u_<username>` |
| Polaris Iceberg | Trino reads Spark-seeded personal and tenant Iceberg tables |
| Query parity | Same portable Iceberg SQL returns the same rows in Spark and Trino |
| Cross-catalog SQL | Trino joins Delta + Iceberg and personal + tenant Iceberg tables |
| Access control | `my` is absent, other user catalog access is blocked, writes are denied |
| Cleanup | Optional Spark cleanup using `DROP TABLE ... PURGE` for Iceberg fixtures |


## Setup

Run this after the docker-compose stack is up and the notebook startup scripts have run. If Polaris credentials were recently rotated, restart `trino` and Spark Connect before running this notebook so long-lived catalog configs are not stale.

In [1]:
import os
import re
from pprint import pprint

from trino.exceptions import TrinoExternalError, TrinoUserError

from berdl_notebook_utils.berdl_settings import get_settings
from berdl_notebook_utils.minio_governance import get_minio_credentials, get_polaris_credentials
from berdl_notebook_utils.setup_spark_session import get_spark_session
from berdl_notebook_utils.setup_trino_session import get_trino_connection

os.environ.setdefault("USER", os.environ.get("NB_USER", ""))
assert os.environ.get("USER"), "USER or NB_USER must be set"

# Make the notebook runnable even when the kernel starts before credential env vars are populated.
# Do not re-provision Polaris if startup already set POLARIS_*; long-lived Trino/Spark
# catalogs cache those credentials and must be restarted after a rotation.
minio_credentials = get_minio_credentials()
if os.environ.get("POLARIS_CREDENTIAL") and os.environ.get("POLARIS_PERSONAL_CATALOG"):
    polaris_credentials = {
        "personal_catalog": os.environ.get("POLARIS_PERSONAL_CATALOG"),
        "tenant_catalogs": [
            c.strip() for c in os.environ.get("POLARIS_TENANT_CATALOGS", "").split(",") if c.strip()
        ],
    }
else:
    polaris_credentials = get_polaris_credentials()
get_settings.cache_clear()
settings = get_settings()


def sanitize_identifier(value):
    return re.sub(r"[^a-z0-9_]", "_", value.lower()).strip("_")

USERNAME = settings.USER
OTHER_USER = "tgu3" if USERNAME != "tgu3" else "tgu2"
LEGACY_CATALOG = f"u_{USERNAME}"
LEGACY_SCHEMA = f"u_{USERNAME}__trino_e2e_delta"
LEGACY_TABLE = "delta_employees"

PERSONAL_CATALOG = sanitize_identifier(USERNAME)
PERSONAL_NAMESPACE = "trino_e2e"
PERSONAL_TABLE = "iceberg_employees"

TENANT_CATALOGS = [c.strip() for c in (settings.POLARIS_TENANT_CATALOGS or "").split(",") if c.strip()]
TENANT_ALIASES = [sanitize_identifier(c.removeprefix("tenant_")) for c in TENANT_CATALOGS]
TENANT_CATALOG = "globalusers" if "globalusers" in TENANT_ALIASES else (TENANT_ALIASES[0] if TENANT_ALIASES else None)
TENANT_NAMESPACE = "trino_e2e"
TENANT_TABLE = "departments"

RUN_CLEANUP = False

print("User:", USERNAME)
print("Legacy Trino catalog:", LEGACY_CATALOG)
print("Personal Iceberg catalog:", PERSONAL_CATALOG)
print("Tenant Iceberg catalog:", TENANT_CATALOG)
print("Polaris personal catalog:", settings.POLARIS_PERSONAL_CATALOG)
print("Polaris tenant catalogs:", settings.POLARIS_TENANT_CATALOGS)


User: tgu2
Legacy Trino catalog: u_tgu2
Personal Iceberg catalog: tgu2
Tenant Iceberg catalog: globalusers
Polaris personal catalog: user_tgu2
Polaris tenant catalogs: tenant_globalusers


In [2]:
results = []


def pass_test(name, detail=""):
    results.append((name, True, detail))
    print(f"✅ {name}" + (f" — {detail}" if detail else ""))


def fail_test(name, detail=""):
    results.append((name, False, detail))
    print(f"❌ {name}" + (f" — {detail}" if detail else ""))


def check(name, condition, detail=""):
    if condition:
        pass_test(name, detail)
    else:
        fail_test(name, detail)


def cursor():
    return conn.cursor()


def trino_rows(sql):
    c = cursor()
    c.execute(sql)
    return [tuple(row) for row in c.fetchall()]


def spark_rows(sql):
    return [tuple(row) for row in spark.sql(sql).collect()]


def expect_trino_ok(name, sql, expected_rows=None):
    try:
        rows = trino_rows(sql)
        detail = f"{len(rows)} rows"
        if expected_rows is not None:
            check(name, rows == expected_rows, f"expected={expected_rows}, got={rows}")
        else:
            pass_test(name, detail)
        return rows
    except Exception as err:
        message = str(err)
        if "unauthorized_client" in message:
            message += " | Restart trino and Spark Connect after Polaris credential rotation."
        fail_test(name, f"{type(err).__name__}: {message}")
        return None


def expect_blocked(name, sql):
    try:
        rows = trino_rows(sql)
        fail_test(name, f"expected blocked, query returned {len(rows)} rows")
    except (TrinoUserError, TrinoExternalError, Exception) as err:
        message = str(err)
        blocked_markers = [
            "ACCESS_DENIED",
            "PERMISSION_DENIED",
            "CATALOG_NOT_FOUND",
            "SCHEMA_NOT_FOUND",
            "TABLE_NOT_FOUND",
            "Cannot create",
            "Cannot insert",
            "Cannot drop",
            "read-only",
            "Access Denied",
        ]
        if any(marker in message for marker in blocked_markers):
            pass_test(name, message[:180])
        else:
            fail_test(name, f"unexpected error: {type(err).__name__}: {message}")


def print_section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


In [3]:
print_section("Connect to Spark and Trino")

spark = get_spark_session(app_name="trino-e2e-seed")
conn = get_trino_connection()

check("Spark session created", spark is not None)
check("Trino connection created", conn is not None)

catalogs = {row[0] for row in trino_rows("SHOW CATALOGS")}
print("Trino catalogs:", sorted(catalogs))
check("Legacy Delta/Hive catalog exists", LEGACY_CATALOG in catalogs, LEGACY_CATALOG)
check("Personal Iceberg catalog exists", PERSONAL_CATALOG in catalogs, PERSONAL_CATALOG)
if TENANT_CATALOG:
    check("Tenant Iceberg catalog exists", TENANT_CATALOG in catalogs, TENANT_CATALOG)
else:
    fail_test("Tenant Iceberg catalog exists", "POLARIS_TENANT_CATALOGS is empty")
check("Spark-only my catalog is absent from Trino", "my" not in catalogs)



Connect to Spark and Trino
✅ Spark session created
✅ Trino connection created
Trino catalogs: ['globalusers', 'tgu2', 'u_tgu2']
✅ Legacy Delta/Hive catalog exists — u_tgu2
✅ Personal Iceberg catalog exists — tgu2
✅ Tenant Iceberg catalog exists — globalusers
✅ Spark-only my catalog is absent from Trino


## Seed Test Data

Spark seeds all mutable fixtures. Trino should only read them.

In [4]:
print_section("Seed Delta and Iceberg fixtures with Spark")

# Legacy Delta fixture in the user's Hive/Delta namespace.
spark.sql(f"CREATE DATABASE IF NOT EXISTS {LEGACY_SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {LEGACY_SCHEMA}.{LEGACY_TABLE}")
spark.sql(
    f"""
    CREATE TABLE {LEGACY_SCHEMA}.{LEGACY_TABLE} (
        id INT,
        name STRING,
        dept STRING,
        salary DOUBLE
    ) USING delta
    """
)
spark.sql(
    f"""
    INSERT INTO {LEGACY_SCHEMA}.{LEGACY_TABLE} VALUES
        (1, 'Ada', 'engineering', 120000.0),
        (2, 'Grace', 'science', 115000.0),
        (3, 'Linus', 'engineering', 118000.0),
        (4, 'Katherine', 'science', 122000.0)
    """
)

# Personal Iceberg fixture. Use Spark's my alias for writes, then refresh the portable alias.
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS my.{PERSONAL_NAMESPACE}")
spark.sql(f"DROP TABLE IF EXISTS my.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE} PURGE")
spark.sql(
    f"""
    CREATE TABLE my.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE} (
        id INT,
        name STRING,
        dept STRING,
        salary DOUBLE
    ) USING iceberg
    """
)
spark.sql(
    f"""
    INSERT INTO my.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE} VALUES
        (1, 'Ada', 'engineering', 120000.0),
        (2, 'Grace', 'science', 115000.0),
        (3, 'Linus', 'engineering', 118000.0),
        (4, 'Katherine', 'science', 122000.0)
    """
)
spark.sql(f"REFRESH TABLE {PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE}")

# Tenant Iceberg fixture.
if TENANT_CATALOG:
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {TENANT_CATALOG}.{TENANT_NAMESPACE}")
    spark.sql(f"DROP TABLE IF EXISTS {TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE} PURGE")
    spark.sql(
        f"""
        CREATE TABLE {TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE} (
            dept_name STRING,
            location STRING,
            headcount INT
        ) USING iceberg
        """
    )
    spark.sql(
        f"""
        INSERT INTO {TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE} VALUES
            ('engineering', 'Oak Ridge', 14),
            ('science', 'Berkeley', 9),
            ('operations', 'Argonne', 5)
        """
    )
    spark.sql(f"REFRESH TABLE {TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE}")

pass_test("Spark seeded Delta fixture", f"{LEGACY_SCHEMA}.{LEGACY_TABLE}")
pass_test("Spark seeded personal Iceberg fixture", f"{PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE}")
if TENANT_CATALOG:
    pass_test("Spark seeded tenant Iceberg fixture", f"{TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE}")



Seed Delta and Iceberg fixtures with Spark
✅ Spark seeded Delta fixture — u_tgu2__trino_e2e_delta.delta_employees
✅ Spark seeded personal Iceberg fixture — tgu2.trino_e2e.iceberg_employees
✅ Spark seeded tenant Iceberg fixture — globalusers.trino_e2e.departments


## Catalog and Schema Discovery

In [5]:
print_section("Catalog and schema discovery")

catalogs = {row[0] for row in trino_rows("SHOW CATALOGS")}
check("Trino exposes legacy catalog", LEGACY_CATALOG in catalogs, LEGACY_CATALOG)
check("Trino exposes personal Polaris catalog", PERSONAL_CATALOG in catalogs, PERSONAL_CATALOG)
check("Trino does not expose Spark-only my alias", "my" not in catalogs)

legacy_schemas = {row[0] for row in trino_rows(f"SHOW SCHEMAS FROM {LEGACY_CATALOG}")}
print("Legacy schemas sample:", sorted(legacy_schemas)[:20])
check("Legacy schema visible", LEGACY_SCHEMA in legacy_schemas, LEGACY_SCHEMA)

personal_schemas = {row[0] for row in trino_rows(f"SHOW SCHEMAS FROM {PERSONAL_CATALOG}")}
print("Personal Iceberg schemas:", sorted(personal_schemas))
check("Personal Iceberg namespace visible", PERSONAL_NAMESPACE in personal_schemas, PERSONAL_NAMESPACE)

if TENANT_CATALOG:
    tenant_schemas = {row[0] for row in trino_rows(f"SHOW SCHEMAS FROM {TENANT_CATALOG}")}
    print("Tenant Iceberg schemas:", sorted(tenant_schemas))
    check("Tenant Iceberg namespace visible", TENANT_NAMESPACE in tenant_schemas, TENANT_NAMESPACE)



Catalog and schema discovery
✅ Trino exposes legacy catalog — u_tgu2
✅ Trino exposes personal Polaris catalog — tgu2
✅ Trino does not expose Spark-only my alias
Legacy schemas sample: ['default', 'globalusers_demo_shared', 'information_schema', 'u_tgu2__demo_personal', 'u_tgu2__trino_e2e_delta']
✅ Legacy schema visible — u_tgu2__trino_e2e_delta
Personal Iceberg schemas: ['information_schema', 'system', 'test_db', 'trino_e2e']
✅ Personal Iceberg namespace visible — trino_e2e
Tenant Iceberg schemas: ['information_schema', 'shared_data', 'system', 'trino_e2e']
✅ Tenant Iceberg namespace visible — trino_e2e


## Read Path Tests

In [6]:
print_section("Read seeded data through Trino")

delta_sql = f"SELECT id, name, dept, salary FROM {LEGACY_CATALOG}.{LEGACY_SCHEMA}.{LEGACY_TABLE} ORDER BY id"
personal_iceberg_sql = f"SELECT id, name, dept, salary FROM {PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE} ORDER BY id"
tenant_iceberg_sql = None
if TENANT_CATALOG:
    tenant_iceberg_sql = f"SELECT dept_name, location, headcount FROM {TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE} ORDER BY dept_name"

delta_rows = expect_trino_ok("Trino reads legacy Delta table", delta_sql)
personal_rows = expect_trino_ok("Trino reads personal Iceberg table", personal_iceberg_sql)
if tenant_iceberg_sql:
    tenant_rows = expect_trino_ok("Trino reads tenant Iceberg table", tenant_iceberg_sql)

agg_rows = expect_trino_ok(
    "Trino aggregate over personal Iceberg",
    f"""
    SELECT dept, count(*) AS employee_count, CAST(sum(salary) AS DOUBLE) AS total_salary
    FROM {PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE}
    GROUP BY dept
    ORDER BY dept
    """,
)
print("Aggregate rows:", agg_rows)



Read seeded data through Trino
✅ Trino reads legacy Delta table — 4 rows
✅ Trino reads personal Iceberg table — 4 rows
✅ Trino reads tenant Iceberg table — 3 rows
✅ Trino aggregate over personal Iceberg — 2 rows
Aggregate rows: [('engineering', 2, 238000.0), ('science', 2, 237000.0)]


## Spark/Trino Portable Query Parity

In [7]:
print_section("Same portable SQL in Spark and Trino")

portable_queries = [
    (
        "Personal Iceberg portable SELECT",
        f"SELECT id, name, dept, salary FROM {PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE} ORDER BY id",
    )
]
if TENANT_CATALOG:
    portable_queries.append(
        (
            "Tenant Iceberg portable SELECT",
            f"SELECT dept_name, location, headcount FROM {TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE} ORDER BY dept_name",
        )
    )
    portable_queries.append(
        (
            "Personal + tenant Iceberg portable JOIN",
            f"""
            SELECT e.id, e.name, e.dept, d.location, d.headcount
            FROM {PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE} e
            LEFT JOIN {TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE} d
              ON e.dept = d.dept_name
            ORDER BY e.id
            """,
        )
    )

for name, sql in portable_queries:
    spark_result = spark_rows(sql)
    trino_result = trino_rows(sql)
    check(name, spark_result == trino_result, f"spark={len(spark_result)} rows, trino={len(trino_result)} rows")



Same portable SQL in Spark and Trino
✅ Personal Iceberg portable SELECT — spark=4 rows, trino=4 rows
✅ Tenant Iceberg portable SELECT — spark=3 rows, trino=3 rows
✅ Personal + tenant Iceberg portable JOIN — spark=4 rows, trino=4 rows


## Trino Cross-Catalog Queries

In [8]:
print_section("Trino cross-catalog joins")

if TENANT_CATALOG:
    delta_iceberg_join = f"""
    SELECT e.id, e.name, e.dept, d.location
    FROM {LEGACY_CATALOG}.{LEGACY_SCHEMA}.{LEGACY_TABLE} e
    LEFT JOIN {TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE} d
      ON e.dept = d.dept_name
    ORDER BY e.id
    """
    rows = expect_trino_ok("Trino joins legacy Delta catalog with tenant Iceberg catalog", delta_iceberg_join)
    print("Join rows:", rows)
else:
    fail_test("Trino joins legacy Delta catalog with tenant Iceberg catalog", "no tenant catalog configured")



Trino cross-catalog joins
✅ Trino joins legacy Delta catalog with tenant Iceberg catalog — 4 rows
Join rows: [(1, 'Ada', 'engineering', 'Oak Ridge'), (2, 'Grace', 'science', 'Berkeley'), (3, 'Linus', 'engineering', 'Oak Ridge'), (4, 'Katherine', 'science', 'Berkeley')]


## Metadata and Information Schema

In [9]:
print_section("Metadata checks")

expect_trino_ok(
    "SHOW COLUMNS from legacy Delta table",
    f"SHOW COLUMNS FROM {LEGACY_CATALOG}.{LEGACY_SCHEMA}.{LEGACY_TABLE}",
)
expect_trino_ok(
    "SHOW COLUMNS from personal Iceberg table",
    f"SHOW COLUMNS FROM {PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE}",
)
expect_trino_ok(
    "Information schema lists personal Iceberg table",
    f"""
    SELECT table_name
    FROM {PERSONAL_CATALOG}.information_schema.tables
    WHERE table_schema = '{PERSONAL_NAMESPACE}' AND table_name = '{PERSONAL_TABLE}'
    """,
    expected_rows=[(PERSONAL_TABLE,)],
)



Metadata checks
✅ SHOW COLUMNS from legacy Delta table — 0 rows
✅ SHOW COLUMNS from personal Iceberg table — 0 rows
✅ Information schema lists personal Iceberg table — expected=[('iceberg_employees',)], got=[('iceberg_employees',)]


[('iceberg_employees',)]

## Access Control and Read-Only Enforcement

In [10]:
print_section("Access control and read-only enforcement")

expect_blocked(
    "Trino cannot use Spark-only my catalog",
    f"SELECT * FROM my.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE}",
)
expect_blocked(
    "Trino blocks other personal Polaris catalog",
    f"SELECT * FROM {OTHER_USER}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE}",
)
expect_blocked(
    "Trino blocks INSERT into personal Iceberg table",
    f"INSERT INTO {PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE} VALUES (99, 'blocked', 'security', 1.0)",
)
expect_blocked(
    "Trino blocks DROP TABLE on personal Iceberg table",
    f"DROP TABLE {PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE}",
)
expect_blocked(
    "Trino blocks CREATE SCHEMA in personal Iceberg catalog",
    f"CREATE SCHEMA {PERSONAL_CATALOG}.trino_e2e_write_blocked",
)
expect_blocked(
    "Trino blocks INSERT into legacy Delta table",
    f"INSERT INTO {LEGACY_CATALOG}.{LEGACY_SCHEMA}.{LEGACY_TABLE} VALUES (99, 'blocked', 'security', 1.0)",
)



Access control and read-only enforcement
✅ Trino cannot use Spark-only my catalog — TrinoUserError(type=USER_ERROR, name=CATALOG_NOT_FOUND, message="line 1:15: Catalog 'my' not found", query_id=20260425_153731_00028_ugbt8)
✅ Trino blocks other personal Polaris catalog — TrinoUserError(type=USER_ERROR, name=CATALOG_NOT_FOUND, message="line 1:15: Catalog 'tgu3' not found", query_id=20260425_153731_00029_ugbt8)
✅ Trino blocks INSERT into personal Iceberg table — TrinoUserError(type=USER_ERROR, name=PERMISSION_DENIED, message="Access Denied: Trino is read-only. Cannot INSERT table 'iceberg_employees'. Use Spark for data modifications.", que
✅ Trino blocks DROP TABLE on personal Iceberg table — TrinoUserError(type=USER_ERROR, name=PERMISSION_DENIED, message="Access Denied: Trino is read-only. Use Spark for table deletion.", query_id=20260425_153731_00031_ugbt8)
✅ Trino blocks CREATE SCHEMA in personal Iceberg catalog — TrinoUserError(type=USER_ERROR, name=PERMISSION_DENIED, message="Access

## Credential Rotation Diagnostic

This cell gives a direct diagnostic for the most common local failure: a Trino dynamic catalog was created with an old Polaris client secret. If this fails with `unauthorized_client`, restart the `trino` container and recreate the Trino connection.

In [11]:
print_section("Credential rotation diagnostic")

try:
    rows = trino_rows(f"SELECT count(*) FROM {PERSONAL_CATALOG}.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE}")
    pass_test("Polaris credential accepted by Trino", rows[0][0])
except Exception as err:
    message = str(err)
    if "unauthorized_client" in message:
        fail_test(
            "Polaris credential accepted by Trino",
            "stale Trino catalog credential; run `docker compose restart trino` and recreate the connection",
        )
    else:
        fail_test("Polaris credential accepted by Trino", f"{type(err).__name__}: {message}")



Credential rotation diagnostic
✅ Polaris credential accepted by Trino — 4


## Optional Cleanup

Set `RUN_CLEANUP = True` in the setup cell and rerun this cell to remove the seeded fixtures. Iceberg cleanup uses `PURGE` so tracked data and metadata files should be removed.

In [12]:
print_section("Cleanup")

if RUN_CLEANUP:
    spark.sql(f"DROP TABLE IF EXISTS {LEGACY_SCHEMA}.{LEGACY_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS my.{PERSONAL_NAMESPACE}.{PERSONAL_TABLE} PURGE")
    if TENANT_CATALOG:
        spark.sql(f"DROP TABLE IF EXISTS {TENANT_CATALOG}.{TENANT_NAMESPACE}.{TENANT_TABLE} PURGE")
    pass_test("Cleanup completed", "seeded tables dropped")
else:
    pass_test("Cleanup skipped", "RUN_CLEANUP is False")



Cleanup
✅ Cleanup skipped — RUN_CLEANUP is False


## Summary

In [13]:
print_section("Trino E2E Summary")

passed = sum(1 for _, ok, _ in results if ok)
failed = sum(1 for _, ok, _ in results if not ok)
print(f"Passed: {passed}")
print(f"Failed: {failed}")

if failed:
    print("\nFailures:")
    for name, ok, detail in results:
        if not ok:
            print(f"- {name}: {detail}")
    raise AssertionError(f"{failed} Trino E2E test(s) failed")

print("All Trino E2E checks passed.")



Trino E2E Summary
Passed: 34
Failed: 0
All Trino E2E checks passed.
